### Step 1: Create Medallion Architecture Schemas
* Creates three separate schemas (`natalkamartinuk55_bronze`, `natalkamartinuk55_silver`, `natalkamartinuk55_gold`) in the shared catalog.
* Uses `MANAGED LOCATION` to point physical Delta storage directly to subdirectories inside the personal ADLS Gen2 container.

In [0]:
%sql
-- 1. Bronze schema
CREATE SCHEMA IF NOT EXISTS dbr_dev_ua5816bd.natalkamartinuk55_bronze
MANAGED LOCATION 'abfss://natalkamartinuk55@dlsua5816bd.dfs.core.windows.net/bronze';

-- 2. Silver schema
CREATE SCHEMA IF NOT EXISTS dbr_dev_ua5816bd.natalkamartinuk55_silver
MANAGED LOCATION 'abfss://natalkamartinuk55@dlsua5816bd.dfs.core.windows.net/silver';

-- 3. Gold schema
CREATE SCHEMA IF NOT EXISTS dbr_dev_ua5816bd.natalkamartinuk55_gold
MANAGED LOCATION 'abfss://natalkamartinuk55@dlsua5816bd.dfs.core.windows.net/gold';


### Step 2: Azure Key Vault Integration (Secret Scope)
* Connects to the Azure Key Vault secret scope (`natalkamartinuk55_scope`).
* Lists available secret keys to verify connectivity between Databricks and Azure Key Vault without exposing plain credentials.

In [0]:

secrets_list = dbutils.secrets.list(scope="natalkamartinuk55_scope")
display(secrets_list)

### Step 3: Validate Secret Retrieval
* Retrieves a test secret (`test`) from Key Vault using `dbutils.secrets.get`.
* Verifies redaction behavior: Databricks automatically masks sensitive secret values as `[REDACTED]` in notebook outputs.

In [0]:
test_secret = dbutils.secrets.get(scope="natalkamartinuk55_scope", key="test")
print(f"Secret read status: {test_secret}")

### Step 4: Legacy Storage Access Pattern (Service Principal)
* Fetches Service Principal credentials (`client_id`, `client_secret`, `tenant_id`) dynamically from the secret scope.
* Prepares OAuth 2.0 parameters for mounting the personal ADLS Gen2 container.
* **Architectural Note:** Direct filesystem mounting (`dbutils.fs.mount`) is legacy and restricted on shared Unity Catalog clusters by design; storage governance is handled natively via Unity Catalog External Locations.

In [0]:

client_id = dbutils.secrets.get(scope="natalkamartinuk55_scope", key="sp-databricks-adls-appid")
client_secret = dbutils.secrets.get(scope="natalkamartinuk55_scope", key="sp-databricks-adls-appkey")
tenant_id = dbutils.secrets.get(scope="natalkamartinuk55_scope", key="tenant-id")


In [0]:
configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

mount_point = "/mnt/natalkamartinuk55_legacy"
storage_account_name = "dlsua5816bd"
container_name = "natalkamartinuk55"


In [0]:

print("Legacy Service Principal credentials successfully loaded from Secret Scope.")
print(f"Target mount point: {mount_point}")
print("Note: Direct dbutils.fs.mount() is disabled on shared UC clusters by design.")

---
##  Architectural Deep Dive (Optional Additions)

### 1. Medallion Architecture Overview
A data design pattern used to logically organize data in a Lakehouse:
* **Bronze Layer (Raw Ingestion):** Stores raw source data as Delta tables with append-only/overwrite patterns, retaining source schema and technical audit metadata.
* **Silver Layer (Cleaned & Conformed):** Enriched, validated, deduplicated, and conformed data suitable for downstream analytics.
* **Gold Layer (Curated Business Level):** Aggregated fact/dimension models and KPIs optimized for BI dashboards, reporting, and ML models.

---

### 2. Managed Identity vs Service Principal

| Dimension | Service Principal (SPN) | Managed Identity (System/User-Assigned) |
| :--- | :--- | :--- |
| **Credential Storage** | Requires managing secrets/passwords (stored in Azure Key Vault). | Zero credential maintenance (automatic key rotation handled by Azure). |
| **Exposure Risk** | Secrets can theoretically leak if misconfigured in code or logs. | Cloud-native token exchange within Azure fabric (no plaintext secrets exist). |
| **Modern Architecture** | Historically used for legacy mounts and external scripts. | Cloud-native best practice for Unity Catalog Storage Credentials and automated Azure resources. |
